Spark streaming - Supply chain data

In [0]:
# Complete PySpark Structured Streaming Pipeline in Databricks
# Supply Chain: Process real-time shipment events from JSON files to Delta table

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("SupplyChainStreaming").getOrCreate()

# Step 1: Define schema for JSON events (supply chain shipments)
schema = StructType([
    StructField("shipment_id", StringType()),
    StructField("status", StringType()),
    StructField("warehouse_id", StringType()),
    StructField("timestamp", TimestampType()),
    StructField("destination", StringType())
])

# Step 2: Read streaming JSON files from input directory
# Drop new JSON files here: /FileStore/streaming_input/
raw_stream = (spark.readStream
    .schema(schema)
    .json("/Volumes/workspace/default/stream_shipment_data/")
)

# Step 3: Business transformations
processed_stream = (raw_stream
    .withColumn("event_hour", hour(col("timestamp")))
    .withColumn("event_date", to_date(col("timestamp")))
    .filter(col("status").isin("shipped", "delivered"))
    .groupBy("warehouse_id", "event_date", "status")
    .agg(count("*").alias("shipment_count"))
)

# Step 4: Write to Delta table with exactly-once guarantees
query = (processed_stream.writeStream
    .format("delta")
    .outputMode("complete")  # For aggregations
    .option("checkpointLocation", "/Volumes/workspace/default/stream_shipment_data/checkpoints/supply_chain/")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table("supply_chain_metrics")
)

# Step 5: Process available data (runs once and stops)
query.awaitTermination()

In [0]:
%sql
SELECT * FROM supply_chain_metrics;


In [0]:
%sql
SELECT * FROM supply_chain_metrics order by event_date asc;


In [0]:
%sql
SELECT * FROM supply_chain_metrics order by event_date asc;


Validate incremental batches for different dates

In [0]:
%sql
SELECT * FROM supply_chain_metrics order by event_date asc;


Validate **complete** mode with incremental batch

In [0]:
%sql
SELECT * FROM supply_chain_metrics order by event_date asc;
